# Module 3: Hybrid Retrieval & Reciprocal Rank Fusion (RRF)

### Why Dense-Only Search Is Not Enough:
- Dense embeddings excel at conceptual, semantic queries (*'how do we handle server crashes'* -> *'circuit breaker pattern'*).
- BUT dense embeddings struggle with exact keywords, serial numbers, codes, or domain acronyms (*'RS256'*, *'CVE-2024-1234'*, *'TTL=60s'*).
- **Solution**: Hybrid search fusing **BM25 (Sparse Lexical)** and **Dense Vector Search**.

### The Reciprocal Rank Fusion (RRF) Formula:
$$\text{RRF Score}(d) = \sum_{m \in M} \frac{1}{k + \text{rank}_m(d)}$$
where $k$ is typically 60. RRF eliminates the scale disparity between unbounded BM25 scores and [0,1] cosine similarities.


In [ ]:
import sys
sys.path.append('..')

from src.core.models import Chunk
from src.indexing.bm25_index import BM25Index
from src.indexing.vector_store import InMemoryVectorStore
from src.embeddings.api_embedder import MockEmbedder
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.sparse_retriever import SparseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever

chunks = [
    Chunk(doc_id='d1', text='External client auth uses asymmetric RS256 JWT tokens with OAuth 2.0.'),
    Chunk(doc_id='d2', text='The circuit breaker trips into OPEN state when failure rate exceeds 50%.'),
    Chunk(doc_id='d3', text='Kafka partitioned topics use replication factor 3 for high availability.'),
]

embedder = MockEmbedder(dimension=64)
vector_store = InMemoryVectorStore()
vector_store.add_chunks(chunks, embedder.embed_texts([c.text for c in chunks]))

bm25 = BM25Index()
bm25.add_chunks(chunks)

dense = DenseRetriever(vector_store, embedder)
sparse = SparseRetriever(bm25)
hybrid = HybridRetriever(dense, sparse, mode='rrf', rrf_k=60)

query = 'What is the RS256 token policy?'
results = hybrid.retrieve(query, top_k=3)
for r in results:
    print(f'Rank {r.rank} | RRF Score: {r.score:.5f} | Snippet: {r.chunk.text}')
